In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# load CSVs (they exist in current working directory)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# convert timestamp to timezone-aware datetime in UTC then convert to Asia/Shanghai
# (assume epoch timestamps are in UTC)
for df in (metric, trace, log, error_logs):
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define incident window in Asia/Shanghai tz
window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)

# 1) metric aggregates
metric_group_count = metric.groupby(['cmdb_id', 'kpi_name'])['value'].count().rename('count')
metric_group_p95 = metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95_value')
metric_agg = pd.concat([metric_group_count, metric_group_p95], axis=1).reset_index()
metric_top = metric_agg.sort_values('count', ascending=False).head(20)

metric_window_count = metric[(metric['dt'] >= window_start) & (metric['dt'] <= window_end)].shape[0]

# 2) trace aggregates
trace_group_count = trace.groupby(['cmdb_id', 'trace_name'])['value'].count().rename('count')
trace_group_p95 = trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('p95_value')
trace_agg = pd.concat([trace_group_count, trace_group_p95], axis=1).reset_index()
trace_top = trace_agg.sort_values('count', ascending=False).head(20)

trace_window_count = trace[(trace['dt'] >= window_start) & (trace['dt'] <= window_end)].shape[0]

# 3) log aggregates
log_group_count = log.groupby(['cmdb_id', 'log_name'])['value'].count().rename('count')
log_group_p95 = log.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.95).rename('p95_value')
log_agg = pd.concat([log_group_count, log_group_p95], axis=1).reset_index()
log_top = log_agg.sort_values('count', ascending=False).head(20)

log_window_count = log[(log['dt'] >= window_start) & (log['dt'] <= window_end)].shape[0]

# 4) error_logs summary
error_total_count = error_logs.shape[0]
error_window_count = error_logs[(error_logs['dt'] >= window_start) & (error_logs['dt'] <= window_end)].shape[0]
error_head5 = error_logs.loc[:, ['timestamp', 'cmdb_id', 'message']].head(5)

# Keep outputs compact: show top combos and window counts and error summary
metric_top, metric_window_count, trace_top, trace_window_count, log_top, log_window_count, error_total_count, error_window_count, error_head5

```
Out[1]:
```
```python
# Summarize the previously computed results as a concise plain-English summary string.
# Reuse variables from the prior execution environment: metric_top, metric_window_count,
# trace_top, trace_window_count, log_top, log_window_count, error_total_count, error_window_count, error_head5

def fmt_row(df, i):
    try:
        r = df.iloc[i]
        return f"{r.iloc[0]} | {r.iloc[1]} | count={int(r['count'])} | p95={r['p95_value']:.6g}"
    except Exception:
        return None

parts = []
parts.append("Summary of telemetry aggregates and incident-window counts (window = 2021-03-24 19:00:00 to 19:30:00 UTC+8):")

# metric summary
m_examples = []
for i in range(min(3, len(metric_top))):
    m_examples.append(fmt_row(metric_top[['cmdb_id','kpi_name','count','p95_value']], i))
parts.append(f"- metric.csv: top examples (cmdb_id | kpi_name | count | p95):")
for ex in m_examples:
    parts.append(f"  - {ex}")
parts.append(f"  Rows within incident window: {metric_window_count}")

# trace summary
t_examples = []
for i in range(min(3, len(trace_top))):
    t_examples.append(fmt_row(trace_top[['cmdb_id','trace_name','count','p95_value']], i))
parts.append(f"- trace.csv: top examples (cmdb_id | trace_name | count | p95):")
for ex in t_examples:
    parts.append(f"  - {ex}")
parts.append(f"  Rows within incident window: {trace_window_count}")

# log summary
l_examples = []
for i in range(min(3, len(log_top))):
    l_examples.append(fmt_row(log_top[['cmdb_id','log_name','count','p95_value']], i))
parts.append(f"- log.csv: top examples (cmdb_id | log_name | count | p95):")
for ex in l_examples:
    parts.append(f"  - {ex}")
parts.append(f"  Rows within incident window: {log_window_count}")

# error_logs summary
parts.append(f"- error_logs.csv: total rows = {error_total_count}, rows in incident window = {error_window_count}")
if error_head5 is None or error_head5.empty:
    parts.append("  No error log samples available (empty).")
else:
    parts.append("  First 5 error log rows (timestamp, cmdb_id, message):")
    # show up to 5 compact lines
    for _, row in error_head5.iterrows():
        parts.append(f"  - {int(row['timestamp'])} | {row['cmdb_id']} | {row['message'][:120]}")

summary = "\n".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     p95_value
1568  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
1569  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
1570  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
1571  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
1572  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
1573  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.680000e+02
1574  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  4.000000e+00
1575  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02
1560  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.231337e+06
1561  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00
1562  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00
1563  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00
1564  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00
1565  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00
1566  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00
1567  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
1557  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  7.000000e+02
1558  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.136860e+05
1559  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.316358e+09
1531  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00, 29276,    cmdb_id                       trace_name  count    p95_value
0     IG01         trace.self.duration_mean     31     0.587074
1     IG01          trace.self.duration_p95     31     3.278200
2     IG01             trace.self.row_count     31  1007.000000
3     IG01  trace.to_Tomcat01.duration_mean     31     0.555889
4     IG01   trace.to_Tomcat01.duration_p95     31     2.942775
5     IG01      trace.to_Tomcat01.row_count     31   140.500000
9     IG01  trace.to_Tomcat03.duration_mean     31     0.751712
12    IG01  trace.to_Tomcat04.duration_mean     31     0.625198
11    IG01      trace.to_Tomcat03.row_count     31   128.500000
10    IG01   trace.to_Tomcat03.duration_p95     31     4.783675
13    IG01   trace.to_Tomcat04.duration_p95     31     2.960500
16    IG02          trace.self.duration_p95     31     3.588875
15    IG02         trace.self.duration_mean     31     0.641885
14    IG01      trace.to_Tomcat04.row_count     31   135.000000
18    IG02  trace.to_Tomcat01.duration_mean     31     0.680809
19    IG02   trace.to_Tomcat01.duration_p95     31     4.655150
20    IG02      trace.to_Tomcat01.row_count     31   135.500000
17    IG02             trace.self.row_count     31  1058.000000
21    IG02  trace.to_Tomcat02.duration_mean     31     0.604279
24    IG02  trace.to_Tomcat03.duration_mean     31     0.759572, 7026,      cmdb_id          log_name  count  p95_value
9   Tomcat02     log.row_count     31   259.5000
7   Tomcat02   log.error_count     31     0.0000
6   Tomcat01     log.row_count     31   267.5000
4   Tomcat01   log.error_count     31     0.0000
18  apache02   log.error_count     31     0.0000
17  apache01     log.row_count     31   497.0000
19  apache02     log.row_count     31   510.5000
16  apache01   log.error_count     31     0.0000
10  Tomcat03   log.error_count     31     0.0000
15  Tomcat04     log.row_count     31   255.5000
13  Tomcat04   log.error_count     31     0.0000
12  Tomcat03     log.row_count     31   256.5000
14  Tomcat04  log.gc_pause_max     12     0.0990
11  Tomcat03  log.gc_pause_max     10     0.0865
5   Tomcat01  log.gc_pause_max     10     0.0800
8   Tomcat02  log.gc_pause_max     10     0.1165
3    Mysql02     log.row_count      1    95.0000
2    Mysql02   log.error_count      1     0.0000
1    Mysql01     log.row_count      1     2.0000
0    Mysql01   log.error_count      1     0.0000, 418, 0, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing variables from the kernel if available; otherwise reload metric.csv and compute thresholds.
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric DF exists
if 'metric' not in globals():
    metric = pd.read_csv('metric.csv')
    metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure global P95 thresholds exist (must be computed from full series BEFORE window filtering)
if 'metric_group_p95' not in globals():
    metric_group_p95 = metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_p95')

# Prepare thresholds DataFrame
metric_thresholds = metric_group_p95.reset_index().rename(columns={0: 'global_p95'}) if isinstance(metric_group_p95, pd.Series) and metric_group_p95.name != 'global_p95' else metric_group_p95.reset_index().rename(columns={metric_group_p95.name: 'global_p95'}) if isinstance(metric_group_p95, pd.Series) else metric_group_p95.copy()
# Ensure column name is 'global_p95'
if 'global_p95' not in metric_thresholds.columns:
    # metric_group_p95 might be a Series named 'p95_value'
    metric_thresholds = metric_group_p95.reset_index().rename(columns={metric_group_p95.name: 'global_p95'})

# Define incident window (UTC+8)
window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)

# Filter metric rows within incident window
metric_window_df = metric[(metric['dt'] >= window_start) & (metric['dt'] <= window_end)].copy()

# Merge window rows with global p95 thresholds (only keep combos that have thresholds)
metric_with_p95 = metric_window_df.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='inner')

# Mark anomalies where value > global_p95
metric_with_p95['is_anom'] = metric_with_p95['value'] > metric_with_p95['global_p95']

# Aggregations per (cmdb_id, kpi_name)
agg = metric_with_p95.groupby(['cmdb_id', 'kpi_name', 'global_p95']).agg(
    anomaly_count = ('is_anom', 'sum'),
    count_in_window = ('value', 'count'),
    max_value_in_window = ('value', 'max'),
    earliest_anom_dt = ('dt', lambda s: s[metric_with_p95.loc[s.index, 'is_anom']].min() if s.size>0 else pd.NaT)
).reset_index()

# Format earliest_anom_dt as string in UTC+8, keep empty if no anomaly (NaT)
agg['earliest_anomaly_timestamp'] = agg['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg.loc[agg['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

# Select and order requested columns
metric_anomalies = agg[['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'count_in_window']].copy()

# Sort by anomaly_count desc and limit to top 20
metric_anomalies_top20 = metric_anomalies.sort_values(['anomaly_count', 'count_in_window'], ascending=[False, False]).head(20)

# Display only the requested compact top-20 summary
metric_anomalies_top20

```
Out[2]:
```
```python
# Create a concise plain-English summary of the metric anomaly detection results.
# Reuse metric_anomalies_top20 variable from the kernel.

if 'metric_anomalies_top20' not in globals():
    raise RuntimeError("metric_anomalies_top20 not found in kernel. Run the anomaly detection step first.")

df = metric_anomalies_top20.reset_index(drop=True)

lines = []
lines.append("Metric anomaly detection summary for incident window 2021-03-24 19:00:00 to 19:30:00 (UTC+8):")
lines.append(f"- Returning top {min(20, len(df))} (cmdb_id, kpi_name) combos sorted by anomaly_count.")
lines.append(f"- Total combos in this top list: {len(df)}.")
# Aggregate by component
comp_counts = df['cmdb_id'].value_counts().head(5)
comp_summary = ", ".join([f"{c}({n})" for c, n in comp_counts.items()])
lines.append(f"- Top affected components (combo count in top list): {comp_summary}")

# Note typical anomaly_count and window counts
if not df.empty:
    most_common_anom = int(df['anomaly_count'].mode()[0])
    common_window_count = int(df['count_in_window'].mode()[0])
    lines.append(f"- Most combos have anomaly_count = {most_common_anom} and count_in_window = {common_window_count} (per top entries).")
else:
    lines.append("- No anomalies detected in the window.")

# List concise top entries (up to 10) with key fields
lines.append("- Top anomaly entries (cmdb_id | abbreviated kpi_name | global_p95 | anomaly_count | earliest_anomaly_timestamp | max_value_in_window | count_in_window):")
for i, row in df.head(10).iterrows():
    k = row['kpi_name']
    k_short = (k[:80] + "...") if len(k) > 80 else k
    gp = f"{row['global_p95']:.6g}"
    ac = int(row['anomaly_count'])
    et = row['earliest_anomaly_timestamp'] if pd.notna(row['earliest_anomaly_timestamp']) else "None"
    mv = f"{row['max_value_in_window']:.6g}"
    cw = int(row['count_in_window'])
    lines.append(f"  - {row['cmdb_id']} | {k_short} | p95={gp} | anomalies={ac} | first={et} | max={mv} | window_rows={cw}")

# Observations: components and metric types
lines.append("- Observations: JVM memory metrics (HeapMemoryUsed) and many Mysql metrics appear repeatedly in the top anomalies, each with 2 points exceeding their global P95 within the 31-row window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  count_in_window
2       IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.280361e+08              2   2021-03-24 19:04:00+0800         8.537210e+08               31
89      IG02  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.003256e+08              2   2021-03-24 19:09:00+0800         8.807143e+08               31
174     MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.009776e+09              2   2021-03-24 19:00:00+0800         1.015766e+09               31
178     MG01   JVM-Threads_7779_JVM_ThreadCount_Threads  7.550000e+01              2   2021-03-24 19:15:00+0800         7.600000e+01               31
275     MG01         OSLinux-OSLinux_ZABBIX_Host_Uptime  7.785036e+06              2   2021-03-24 19:29:00+0800         7.785126e+06               31
278     MG02  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.162090e+08              2   2021-03-24 19:05:00+0800         9.371192e+08               31
380     MG02         OSLinux-OSLinux_ZABBIX_Host_Uptime  7.784445e+06              2   2021-03-24 19:29:00+0800         7.784536e+06               31
384  Mysql01          Mysql-MySQL_3306_Binlog Cache Use  4.806668e+06              2   2021-03-24 19:25:00+0800         4.806969e+06               31
388  Mysql01                Mysql-MySQL_3306_Bytes Sent  7.905929e+10              2   2021-03-24 19:25:00+0800         7.906353e+10               31
396  Mysql01                Mysql-MySQL_3306_Com Select  1.059414e+08              2   2021-03-24 19:25:00+0800         1.059476e+08               31
397  Mysql01                Mysql-MySQL_3306_Com Update  4.586991e+06              2   2021-03-24 19:25:00+0800         4.587292e+06               31
399  Mysql01               Mysql-MySQL_3306_Connections  1.433738e+06              2   2021-03-24 19:25:00+0800         1.433756e+06               31
400  Mysql01   Mysql-MySQL_3306_Created Tmp Disk Tables  2.769185e+05              2   2021-03-24 19:25:00+0800         2.769230e+05               31
402  Mysql01        Mysql-MySQL_3306_Created Tmp Tables  9.778250e+05              2   2021-03-24 19:25:00+0800         9.778430e+05               31
406  Mysql01            Mysql-MySQL_3306_Handler Commit  1.229499e+08              2   2021-03-24 19:25:00+0800         1.229573e+08               31
408  Mysql01        Mysql-MySQL_3306_Handler Read First  6.304004e+06              2   2021-03-24 19:25:00+0800         6.304377e+06               31
409  Mysql01          Mysql-MySQL_3306_Handler Read Key  1.403059e+08              2   2021-03-24 19:25:00+0800         1.403134e+08               31
413  Mysql01     Mysql-MySQL_3306_Handler Read Rnd Next  3.658439e+09              2   2021-03-24 19:25:00+0800         3.658644e+09               31
417  Mysql01            Mysql-MySQL_3306_Handler Update  1.584400e+07              2   2021-03-24 19:25:00+0800         1.584430e+07               31
418  Mysql01             Mysql-MySQL_3306_Handler Write  2.075722e+08              2   2021-03-24 19:25:00+0800         2.075762e+08               31```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace DataFrame and full-series P95 thresholds exist; otherwise load/compute them
if 'trace' not in globals():
    trace = pd.read_csv('trace.csv')

# Ensure timezone-aware datetime column
if 'dt' not in trace.columns:
    trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute full-series global P95 thresholds if not already computed
if 'trace_group_p95' not in globals():
    trace_group_p95 = trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('global_p95')

# Prepare thresholds DataFrame
trace_thresholds = trace_group_p95.reset_index().rename(columns={trace_group_p95.name: 'global_p95'})

# Define incident window in UTC+8
window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)

# 1) Filter trace rows within the incident window
trace_window_df = trace[(trace['dt'] >= window_start) & (trace['dt'] <= window_end)].copy()

# 2) Merge window rows with precomputed global P95 thresholds (inner join ensures only combos with thresholds)
trace_with_p95 = trace_window_df.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='inner')

# Mark anomalies where value > global_p95
trace_with_p95['is_anom'] = trace_with_p95['value'] > trace_with_p95['global_p95']

# Aggregations per (cmdb_id, trace_name)
agg_main = trace_with_p95.groupby(['cmdb_id', 'trace_name', 'global_p95']).agg(
    anomaly_count = ('is_anom', 'sum'),
    count_in_window = ('value', 'count'),
    max_value_in_window = ('value', 'max')
).reset_index()

# Earliest anomaly timestamp per group (if any)
earliest = (
    trace_with_p95[trace_with_p95['is_anom']]
    .groupby(['cmdb_id', 'trace_name'])['dt']
    .min()
    .reset_index()
    .rename(columns={'dt': 'earliest_anom_dt'})
)

# Merge earliest timestamps
agg = agg_main.merge(earliest, on=['cmdb_id', 'trace_name'], how='left')

# Format earliest timestamp as string in UTC+8, set None if missing
agg['earliest_anomaly_timestamp'] = agg['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg.loc[agg['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

# Select requested columns and filter anomaly_count > 0
trace_anomalies = agg[
    ['cmdb_id', 'trace_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'count_in_window']
].copy()

trace_anomalies_pos = trace_anomalies[trace_anomalies['anomaly_count'] > 0]

# 3) Return top 20 combos sorted by anomaly_count desc (then by count_in_window desc)
trace_anomalies_top20 = trace_anomalies_pos.sort_values(['anomaly_count', 'count_in_window'], ascending=[False, False]).head(20)

# Keep output compact
trace_anomalies_top20

```
Out[3]:
```
```python
# Summarize trace anomaly detection results as a plain-English string.
# Reuse existing variable trace_anomalies_top20 from the kernel.

df = trace_anomalies_top20 if 'trace_anomalies_top20' in globals() else None
if df is None or df.empty:
    summary = "No trace anomalies with anomaly_count > 0 were found in the incident window."
else:
    lines = []
    lines.append("Trace anomaly detection summary for 2021-03-24 19:00:00 to 19:30:00 (UTC+8):")
    lines.append(f"- Top combos returned: {len(df)} (only combos with anomaly_count > 0).")
    comp_counts = df['cmdb_id'].value_counts()
    top_comps = ", ".join([f"{c}({n})" for c, n in comp_counts.items()])
    lines.append(f"- Affected components in top list: {top_comps}")
    # Typical anomaly pattern
    common_anom = int(df['anomaly_count'].mode().iloc[0])
    common_window = int(df['count_in_window'].mode().iloc[0])
    lines.append(f"- Typical pattern: anomaly_count = {common_anom} and count_in_window = {common_window} for most top entries.")
    lines.append("- Examples (cmdb_id | trace_name | global_p95 | anomalies | first_anomaly | max_in_window | window_rows):")
    for _, row in df.head(10).iterrows():
        tn = row['trace_name']
        tn_short = (tn[:80] + "...") if len(tn) > 80 else tn
        gp = f"{row['global_p95']:.6g}"
        ac = int(row['anomaly_count'])
        et = row['earliest_anomaly_timestamp'] if pd.notna(row['earliest_anomaly_timestamp']) else "None"
        mv = f"{row['max_value_in_window']:.6g}"
        cw = int(row['count_in_window'])
        lines.append(f"  - {row['cmdb_id']} | {tn_short} | p95={gp} | anomalies={ac} | first={et} | max={mv} | rows={cw}")
    lines.append("- Observations: IG01 and IG02 dominate the top anomalies. Most trace metrics (duration_mean, duration_p95, row_count, and 'to_TomcatX' traces) each had 2 values exceeding their global P95 during the 31-minute window; earliest anomaly times cluster around 19:03:00+0800 (some at 19:00, 19:02, 19:15, or 19:23).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name   global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  count_in_window
0     IG01         trace.self.duration_mean     0.587074              2   2021-03-24 19:03:00+0800             0.720301               31
1     IG01          trace.self.duration_p95     3.278200              2   2021-03-24 19:03:00+0800             6.041000               31
2     IG01             trace.self.row_count  1007.000000              2   2021-03-24 19:00:00+0800          1052.000000               31
3     IG01  trace.to_Tomcat01.duration_mean     0.555889              2   2021-03-24 19:03:00+0800             0.765915               31
4     IG01   trace.to_Tomcat01.duration_p95     2.942775              2   2021-03-24 19:03:00+0800             6.079800               31
5     IG01      trace.to_Tomcat01.row_count   140.500000              2   2021-03-24 19:00:00+0800           144.000000               31
9     IG01  trace.to_Tomcat03.duration_mean     0.751712              2   2021-03-24 19:03:00+0800             1.089500               31
10    IG01   trace.to_Tomcat03.duration_p95     4.783675              2   2021-03-24 19:03:00+0800             6.092600               31
11    IG01      trace.to_Tomcat03.row_count   128.500000              2   2021-03-24 19:19:00+0800           132.000000               31
12    IG01  trace.to_Tomcat04.duration_mean     0.625198              2   2021-03-24 19:15:00+0800             0.754966               31
13    IG01   trace.to_Tomcat04.duration_p95     2.960500              2   2021-03-24 19:15:00+0800             5.459000               31
15    IG02         trace.self.duration_mean     0.641885              2   2021-03-24 19:03:00+0800             0.770723               31
16    IG02          trace.self.duration_p95     3.588875              2   2021-03-24 19:03:00+0800             6.895200               31
17    IG02             trace.self.row_count  1058.000000              2   2021-03-24 19:02:00+0800          1100.000000               31
18    IG02  trace.to_Tomcat01.duration_mean     0.680809              2   2021-03-24 19:03:00+0800             1.036029               31
19    IG02   trace.to_Tomcat01.duration_p95     4.655150              2   2021-03-24 19:03:00+0800             8.974700               31
20    IG02      trace.to_Tomcat01.row_count   135.500000              2   2021-03-24 19:23:00+0800           139.000000               31
21    IG02  trace.to_Tomcat02.duration_mean     0.604279              2   2021-03-24 19:03:00+0800             0.681112               31
22    IG02   trace.to_Tomcat02.duration_p95     3.036850              2   2021-03-24 19:03:00+0800             4.098300               31
23    IG02      trace.to_Tomcat02.row_count   137.500000              2   2021-03-24 19:02:00+0800           146.000000               31```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse timezone and metric DataFrame if available
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric and full-series thresholds exist
if 'metric' not in globals():
    metric = pd.read_csv('metric.csv')
    metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if 'dt' not in metric.columns:
        metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute full-series global P95 thresholds if missing (must be computed before window filtering)
if 'metric_group_p95' not in globals():
    metric_group_p95 = metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_p95')

metric_thresholds = metric_group_p95.reset_index().rename(columns={metric_group_p95.name: 'global_p95'})

# Determine Mysql01 KPI list from metric_anomalies_top20 if available; otherwise derive from thresholds
if 'metric_anomalies_top20' in globals():
    mysql_kpis = metric_anomalies_top20.loc[metric_anomalies_top20['cmdb_id']=='Mysql01', 'kpi_name'].unique().tolist()
else:
    # fallback: take KPIs for Mysql01 that exist in thresholds (limited to 20)
    mysql_kpis = metric_thresholds.loc[metric_thresholds['cmdb_id']=='Mysql01','kpi_name'].unique().tolist()

# limit to at most 20 KPIs
mysql_kpis = mysql_kpis[:20]

# Define incident window in UTC+8
window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)

# Filter metric rows to the incident window (use full metric df dt)
metric_window_df = metric[(metric['dt'] >= window_start) & (metric['dt'] <= window_end)].copy()

# Filter to Mysql01 and selected KPIs
metric_window_mysql = metric_window_df[(metric_window_df['cmdb_id']=='Mysql01') & (metric_window_df['kpi_name'].isin(mysql_kpis))].copy()

# Merge with precomputed global p95 thresholds
metric_window_mysql = metric_window_mysql.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies where value > global_p95
metric_window_mysql['is_anom'] = metric_window_mysql['value'] > metric_window_mysql['global_p95']

# Per-KPI aggregations
agg = metric_window_mysql.groupby(['kpi_name','global_p95'], dropna=False).agg(
    anomaly_count = ('is_anom', 'sum'),
    count_in_window = ('value', 'count'),
    max_value_in_window = ('value', 'max')
).reset_index()

# earliest anomaly timestamp per kpi (if any)
earliest = (
    metric_window_mysql[metric_window_mysql['is_anom']]
    .groupby('kpi_name')['dt']
    .min()
    .reset_index()
    .rename(columns={'dt':'earliest_anom_dt'})
)

agg = agg.merge(earliest, on='kpi_name', how='left')
agg['earliest_anomaly_timestamp'] = agg['earliest_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg.loc[agg['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

# Keep requested columns and order, limit to KPIs found (and at most 20)
summary_cols = ['kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','count_in_window']
mysql_summary = agg[summary_cols].sort_values(['anomaly_count','count_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# For raw per-kpi samples: up to first 6 window rows per KPI sorted by timestamp
# Keep only timestamp (epoch) and formatted dt and value for compactness
samples = (
    metric_window_mysql.sort_values(['kpi_name','dt'])
    .groupby('kpi_name', group_keys=False)
    .apply(lambda d: d[['kpi_name','timestamp','dt','value']].head(6))
    .reset_index(drop=True)
)
# Format dt for readability
samples['dt_str'] = samples['dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
samples_display = samples[['kpi_name','timestamp','dt_str','value']]

# Final compact outputs: mysql_summary and samples_display
mysql_summary, samples_display

```
Out[4]:
```
```python
# Summarize Mysql01 KPI anomalies and show up to 6 sample window rows per KPI (compact plain-English).
# Reuse kernel variables: mysql_summary, samples_display

import pandas as pd

if 'mysql_summary' not in globals() or 'samples_display' not in globals():
    raise RuntimeError("Required variables mysql_summary or samples_display not found. Run previous steps first.")

df_sum = mysql_summary.copy()
df_samples = samples_display.copy()

lines = []
lines.append("Mysql01 metric anomaly summary (window 2021-03-24 19:00:00 to 19:30:00 UTC+8):")
lines.append(f"- KPIs included: {len(df_sum)} (limited to top 20).")
lines.append("- Per-KPI summary: kpi_name | global_p95 | anomaly_count | earliest_anomaly_timestamp | max_value_in_window | count_in_window")

for _, r in df_sum.iterrows():
    k = r['kpi_name']
    gp = f"{r['global_p95']:.6g}"
    ac = int(r['anomaly_count'])
    et = r['earliest_anomaly_timestamp'] if pd.notna(r['earliest_anomaly_timestamp']) else "None"
    mv = f"{r['max_value_in_window']:.6g}" if pd.notna(r['max_value_in_window']) else "None"
    cw = int(r['count_in_window'])
    lines.append(f"  - {k} | p95={gp} | anomalies={ac} | first={et} | max={mv} | rows={cw}")

lines.append("- Sample window rows (up to 6 per KPI) to inspect anomaly continuity (timestamp epoch | local dt | value):")

# For each KPI in the summary, list up to 6 sample rows
for k in df_sum['kpi_name'].tolist():
    samp = df_samples[df_samples['kpi_name']==k].head(6)
    lines.append(f"  * {k}:")
    if samp.empty:
        lines.append("    - (no window rows found)")
        continue
    for _, s in samp.iterrows():
        ts = int(s['timestamp'])
        dt = s['dt_str']
        val = f"{s['value']:.6g}"
        lines.append(f"    - {ts} | {dt} | {val}")

lines.append("- Observation: Each listed Mysql01 KPI had 2 points exceeding its global P95 within the 31-row window; anomalous points appear around 19:25:00+0800 and sample rows show these high values among otherwise lower values (check per-KPI samples above for consecutiveness).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                    kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  count_in_window
0          Mysql-MySQL_3306_Binlog Cache Use  4.806668e+06              2   2021-03-24 19:25:00+0800         4.806969e+06               31
1                Mysql-MySQL_3306_Bytes Sent  7.905929e+10              2   2021-03-24 19:25:00+0800         7.906353e+10               31
2                Mysql-MySQL_3306_Com Select  1.059414e+08              2   2021-03-24 19:25:00+0800         1.059476e+08               31
3                Mysql-MySQL_3306_Com Update  4.586991e+06              2   2021-03-24 19:25:00+0800         4.587292e+06               31
4               Mysql-MySQL_3306_Connections  1.433738e+06              2   2021-03-24 19:25:00+0800         1.433756e+06               31
5   Mysql-MySQL_3306_Created Tmp Disk Tables  2.769185e+05              2   2021-03-24 19:25:00+0800         2.769230e+05               31
6        Mysql-MySQL_3306_Created Tmp Tables  9.778250e+05              2   2021-03-24 19:25:00+0800         9.778430e+05               31
7            Mysql-MySQL_3306_Handler Commit  1.229499e+08              2   2021-03-24 19:25:00+0800         1.229573e+08               31
8        Mysql-MySQL_3306_Handler Read First  6.304004e+06              2   2021-03-24 19:25:00+0800         6.304377e+06               31
9          Mysql-MySQL_3306_Handler Read Key  1.403059e+08              2   2021-03-24 19:25:00+0800         1.403134e+08               31
10    Mysql-MySQL_3306_Handler Read Rnd Next  3.658439e+09              2   2021-03-24 19:25:00+0800         3.658644e+09               31
11           Mysql-MySQL_3306_Handler Update  1.584400e+07              2   2021-03-24 19:25:00+0800         1.584430e+07               31
12            Mysql-MySQL_3306_Handler Write  2.075722e+08              2   2021-03-24 19:25:00+0800         2.075762e+08               31,                              kpi_name   timestamp                    dt_str         value
0   Mysql-MySQL_3306_Binlog Cache Use  1616583600  2021-03-24 19:00:00+0800  1.783300e+00
1   Mysql-MySQL_3306_Binlog Cache Use  1616583660  2021-03-24 19:01:00+0800  4.805268e+06
2   Mysql-MySQL_3306_Binlog Cache Use  1616583720  2021-03-24 19:02:00+0800  1.050000e+00
3   Mysql-MySQL_3306_Binlog Cache Use  1616583780  2021-03-24 19:03:00+0800  1.183300e+00
4   Mysql-MySQL_3306_Binlog Cache Use  1616583840  2021-03-24 19:04:00+0800  4.805459e+06
..                                ...         ...                       ...           ...
73     Mysql-MySQL_3306_Handler Write  1616583660  2021-03-24 19:01:00+0800  2.075524e+08
74     Mysql-MySQL_3306_Handler Write  1616583720  2021-03-24 19:02:00+0800  1.468330e+01
75     Mysql-MySQL_3306_Handler Write  1616583780  2021-03-24 19:03:00+0800  1.468330e+01
76     Mysql-MySQL_3306_Handler Write  1616583840  2021-03-24 19:04:00+0800  2.075550e+08
77     Mysql-MySQL_3306_Handler Write  1616583900  2021-03-24 19:05:00+0800  1.468330e+01

[78 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables where possible
tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist; otherwise reconstruct from files
if 'mysql_summary' not in globals():
    raise RuntimeError("mysql_summary not found. Run prior steps to create mysql_summary.")

# Determine KPI list from mysql_summary
kpi_list = mysql_summary['kpi_name'].tolist()[:20]

# Ensure metric_window_mysql exists (windowed rows for Mysql01 KPIs merged with thresholds and is_anom)
if 'metric_window_mysql' not in globals():
    # Rebuild metric_window_mysql using prior assumptions/variables
    if 'metric' not in globals():
        metric = pd.read_csv('metric.csv')
        metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    if 'metric_group_p95' not in globals():
        metric_group_p95 = metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_p95')
    metric_thresholds = metric_group_p95.reset_index().rename(columns={metric_group_p95.name: 'global_p95'})
    window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
    window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)
    metric_window_df = metric[(metric['dt'] >= window_start) & (metric['dt'] <= window_end)].copy()
    metric_window_mysql = metric_window_df[(metric_window_df['cmdb_id']=='Mysql01') & (metric_window_df['kpi_name'].isin(kpi_list))].merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    metric_window_mysql['is_anom'] = metric_window_mysql['value'] > metric_window_mysql['global_p95']

# Prepare rows for fault-identification table
rows = []
for kpi in kpi_list:
    grp = metric_window_mysql[metric_window_mysql['kpi_name']==kpi].sort_values('dt')
    if grp.empty:
        # No data in window for this KPI; still include with zeros/None
        rows.append({
            'kpi_name': kpi,
            'global_p95': None,
            'anomaly_count': 0,
            'earliest_anomaly_timestamp': None,
            'max_value_in_window': None,
            'count_in_window': 0,
            'percent_breach': None,
            'consecutive_flag': False,
            'consecutive_run_len': 0
        })
        continue

    global_p95 = grp['global_p95'].iloc[0]
    anomaly_mask = grp['is_anom'].astype(bool).values
    anomaly_count = int(anomaly_mask.sum())
    max_value = grp['value'].max()
    count_in_window = int(grp.shape[0])

    # earliest anomaly timestamp
    if anomaly_count > 0:
        earliest_dt = grp.loc[grp['is_anom'], 'dt'].min()
        earliest_ts = earliest_dt.strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        earliest_ts = None

    # percent breach: (max - p95) / p95 * 100
    if pd.notna(global_p95) and global_p95 != 0:
        percent_breach = (max_value - global_p95) / global_p95 * 100.0
    else:
        percent_breach = np.nan

    # compute longest consecutive run of True in anomaly_mask
    if anomaly_mask.size == 0:
        longest_run = 0
    else:
        longest_run = 0
        current = 0
        for v in anomaly_mask:
            if v:
                current += 1
                if current > longest_run:
                    longest_run = current
            else:
                current = 0

    consecutive_flag = longest_run >= 2

    rows.append({
        'kpi_name': kpi,
        'global_p95': global_p95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'max_value_in_window': max_value,
        'count_in_window': count_in_window,
        'percent_breach': percent_breach,
        'consecutive_flag': consecutive_flag,
        'consecutive_run_len': int(longest_run)
    })

# Assemble DataFrame and keep it compact (limit to 20 rows)
mysql_fault_table = pd.DataFrame(rows)[[
    'kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp',
    'max_value_in_window','count_in_window','percent_breach','consecutive_flag','consecutive_run_len'
]].head(20)

# Round numeric columns for compactness
mysql_fault_table['global_p95'] = mysql_fault_table['global_p95'].map(lambda x: np.nan if pd.isna(x) else float(x))
mysql_fault_table['max_value_in_window'] = mysql_fault_table['max_value_in_window'].map(lambda x: np.nan if pd.isna(x) else float(x))
mysql_fault_table['percent_breach'] = mysql_fault_table['percent_breach'].map(lambda x: np.nan if pd.isna(x) else round(x, 3))

# Display the compact fault-identification table (up to 20 rows)
mysql_fault_table

```
Out[5]:
```
```python
# Summarize Mysql01 fault-identification results in plain English (compact).
# Reuse existing kernel variable: mysql_fault_table

if 'mysql_fault_table' not in globals():
    raise RuntimeError("mysql_fault_table not found in kernel. Run the previous analysis steps first.")

df = mysql_fault_table.copy()

n_kpis = df.shape[0]
anom_counts = sorted(df['anomaly_count'].unique())
earliest_vals = df['earliest_anomaly_timestamp'].dropna().unique().tolist()
pct_min = df['percent_breach'].dropna().min()
pct_max = df['percent_breach'].dropna().max()
consec_any = df['consecutive_flag'].any()
max_run = int(df['consecutive_run_len'].max())

summary_lines = []
summary_lines.append(f"Mysql01 fault-identification summary for 2021-03-24 19:00–19:30 (UTC+8):")
summary_lines.append(f"- KPIs analyzed: {n_kpis}.")
summary_lines.append(f"- Anomaly counts observed: distinct values = {anom_counts} (most KPIs have 2 breaches in the window).")
if len(earliest_vals) == 0:
    summary_lines.append("- No earliest anomaly timestamps recorded.")
else:
    # show common earliest time(s)
    summary_lines.append(f"- Earliest anomaly timestamps (per KPI) observed; examples include: {earliest_vals[:3]}.")
summary_lines.append(f"- Percent breach range (max vs p95): {pct_min:.3f}% to {pct_max:.3f}% (very small breaches, ~0.001%–0.007%).")
if consec_any:
    summary_lines.append(f"- At least one KPI showed consecutive breaches; longest consecutive run across KPIs = {max_run}.")
else:
    summary_lines.append(f"- No KPI showed consecutive breaches >=2; longest consecutive run across KPIs = {max_run} (breaches are isolated spikes).")
summary_lines.append("- Interpretation: All listed Mysql01 KPIs show 2 isolated points above their full-series P95 around ~19:25+08:00, with very small percentage increases over P95. This pattern indicates transient spikes rather than sustained degradation; investigate transient activity at ~19:25 (queries/batches/backups/replication) but immediate persistent fault is unlikely based on these metrics.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  count_in_window  percent_breach  consecutive_flag  consecutive_run_len
0          Mysql-MySQL_3306_Binlog Cache Use  4.806668e+06              2   2021-03-24 19:25:00+0800         4.806969e+06               31           0.006             False                    1
1                Mysql-MySQL_3306_Bytes Sent  7.905929e+10              2   2021-03-24 19:25:00+0800         7.906353e+10               31           0.005             False                    1
2                Mysql-MySQL_3306_Com Select  1.059414e+08              2   2021-03-24 19:25:00+0800         1.059476e+08               31           0.006             False                    1
3                Mysql-MySQL_3306_Com Update  4.586991e+06              2   2021-03-24 19:25:00+0800         4.587292e+06               31           0.007             False                    1
4               Mysql-MySQL_3306_Connections  1.433738e+06              2   2021-03-24 19:25:00+0800         1.433756e+06               31           0.001             False                    1
5   Mysql-MySQL_3306_Created Tmp Disk Tables  2.769185e+05              2   2021-03-24 19:25:00+0800         2.769230e+05               31           0.002             False                    1
6        Mysql-MySQL_3306_Created Tmp Tables  9.778250e+05              2   2021-03-24 19:25:00+0800         9.778430e+05               31           0.002             False                    1
7            Mysql-MySQL_3306_Handler Commit  1.229499e+08              2   2021-03-24 19:25:00+0800         1.229573e+08               31           0.006             False                    1
8        Mysql-MySQL_3306_Handler Read First  6.304004e+06              2   2021-03-24 19:25:00+0800         6.304377e+06               31           0.006             False                    1
9          Mysql-MySQL_3306_Handler Read Key  1.403059e+08              2   2021-03-24 19:25:00+0800         1.403134e+08               31           0.005             False                    1
10    Mysql-MySQL_3306_Handler Read Rnd Next  3.658439e+09              2   2021-03-24 19:25:00+0800         3.658644e+09               31           0.006             False                    1
11           Mysql-MySQL_3306_Handler Update  1.584400e+07              2   2021-03-24 19:25:00+0800         1.584430e+07               31           0.002             False                    1
12            Mysql-MySQL_3306_Handler Write  2.075722e+08              2   2021-03-24 19:25:00+0800         2.075762e+08               31           0.002             False                    1```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed globals where possible:
# trace_anomalies_top20, metric_anomalies_top20, mysql_fault_table,
# trace_with_p95, metric_with_p95, metric_window_mysql

# Ensure required variables exist (recompute minimal pieces if absent)
if 'trace_anomalies_top20' not in globals():
    raise RuntimeError("trace_anomalies_top20 not found; run prior trace anomaly step first.")
if 'metric_anomalies_top20' not in globals():
    raise RuntimeError("metric_anomalies_top20 not found; run prior metric anomaly step first.")
if 'mysql_fault_table' not in globals():
    raise RuntimeError("mysql_fault_table not found; run prior mysql fault step first.")

# Ensure trace_with_p95 exists (window rows merged with global p95)
if 'trace_with_p95' not in globals():
    # try to build from trace_window_df and trace_group_p95
    if 'trace' not in globals() or 'trace_group_p95' not in globals():
        trace = pd.read_csv('trace.csv')
        trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        trace_group_p95 = trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('global_p95')
    trace_thresholds = trace_group_p95.reset_index().rename(columns={trace_group_p95.name: 'global_p95'})
    window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
    window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)
    trace_window_df = trace[(trace['dt'] >= window_start) & (trace['dt'] <= window_end)].copy()
    trace_with_p95 = trace_window_df.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='inner')
    trace_with_p95['is_anom'] = trace_with_p95['value'] > trace_with_p95['global_p95']

# Ensure metric_with_p95 exists
if 'metric_with_p95' not in globals():
    # rebuild minimal metric_with_p95 if possible
    if 'metric' not in globals() or 'metric_group_p95' not in globals():
        metric = pd.read_csv('metric.csv')
        metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        metric_group_p95 = metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_p95')
    metric_thresholds = metric_group_p95.reset_index().rename(columns={metric_group_p95.name: 'global_p95'})
    window_start = pd.Timestamp('2021-03-24 19:00:00', tz=tz)
    window_end = pd.Timestamp('2021-03-24 19:30:00', tz=tz)
    metric_window_df = metric[(metric['dt'] >= window_start) & (metric['dt'] <= window_end)].copy()
    metric_with_p95 = metric_window_df.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='inner')
    metric_with_p95['is_anom'] = metric_with_p95['value'] > metric_with_p95['global_p95']

# Helper to compute longest consecutive run length of True in time-ordered boolean series
def longest_consecutive_run(bool_series):
    if len(bool_series) == 0:
        return 0
    longest = 0
    cur = 0
    for v in bool_series:
        if v:
            cur += 1
            if cur > longest:
                longest = cur
        else:
            cur = 0
    return int(longest)

# 1) Prepare trace candidates: take all rows from trace_anomalies_top20
trace_cands = trace_anomalies_top20.copy()
trace_cands = trace_cands.rename(columns={'trace_name':'name'})
trace_cands['source_type'] = 'trace'
trace_cands['cmdb_id'] = trace_cands['cmdb_id'].astype(str)

# Compute consecutive_run_len for each trace candidate using trace_with_p95 windowed data
trace_runs = []
for _, r in trace_cands.iterrows():
    cm = r['cmdb_id']
    name = r['name']
    grp = trace_with_p95[(trace_with_p95['cmdb_id']==cm) & (trace_with_p95['trace_name']==name)].sort_values('dt')
    run_len = longest_consecutive_run(grp['is_anom'].astype(bool).values)
    trace_runs.append(run_len)
trace_cands = trace_cands.assign(consecutive_run_len = trace_runs)

# 2) Prepare metric HeapMemoryUsed candidates from metric_anomalies_top20 for specified cmdb_ids
cm_list = ['IG01','IG02','MG01','MG02']
metric_heap = metric_anomalies_top20[
    (metric_anomalies_top20['cmdb_id'].isin(cm_list)) &
    (metric_anomalies_top20['kpi_name'].str.contains('HeapMemoryUsed', na=False))
].copy()

# Rename and set source_type
metric_heap = metric_heap.rename(columns={'kpi_name':'name'})
metric_heap['source_type'] = 'metric'
metric_heap['cmdb_id'] = metric_heap['cmdb_id'].astype(str)

# Compute consecutive_run_len for metric heap candidates using metric_with_p95
metric_heap_runs = []
for _, r in metric_heap.iterrows():
    cm = r['cmdb_id']
    name = r['name']
    grp = metric_with_p95[(metric_with_p95['cmdb_id']==cm) & (metric_with_p95['kpi_name']==name)].sort_values('dt')
    run_len = longest_consecutive_run(grp['is_anom'].astype(bool).values)
    metric_heap_runs.append(run_len)
metric_heap = metric_heap.assign(consecutive_run_len = metric_heap_runs)

# 3) Prepare Mysql01 KPIs from mysql_fault_table for contrast
mysql_cands = mysql_fault_table.copy()
mysql_cands = mysql_cands.rename(columns={'kpi_name':'name', 'percent_breach':'percent_breach_existing'})
mysql_cands['source_type'] = 'metric'
mysql_cands['cmdb_id'] = 'Mysql01'
# mysql_fault_table already has consecutive_run_len and percent_breach computed; align column names
mysql_cands = mysql_cands.rename(columns={'max_value_in_window':'max_value_in_window', 'anomaly_count':'anomaly_count', 'earliest_anomaly_timestamp':'earliest_anomaly_timestamp'})
# Ensure global_p95 column present: join with metric thresholds if missing
if 'global_p95' not in mysql_cands.columns or mysql_cands['global_p95'].isnull().all():
    if 'metric_group_p95' in globals():
        thr = metric_group_p95.reset_index().rename(columns={metric_group_p95.name:'global_p95'})
        mysql_cands = mysql_cands.merge(thr[thr['cmdb_id']=='Mysql01'][['kpi_name','global_p95']].rename(columns={'kpi_name':'name'}), on='name', how='left')
# compute percent_breach (prefer precomputed percent_breach if present)
mysql_percent = []
for _, r in mysql_cands.iterrows():
    gp = r.get('global_p95', np.nan)
    mv = r.get('max_value_in_window', np.nan)
    if pd.notna(gp) and gp != 0:
        pb = (mv - gp) / gp * 100.0
    else:
        pb = np.nan
    mysql_percent.append(pb)
mysql_cands['percent_breach'] = mysql_percent
# ensure consecutive_run_len exists (it was computed earlier)
if 'consecutive_run_len' not in mysql_cands.columns:
    mysql_cands['consecutive_run_len'] = mysql_fault_table.get('consecutive_run_len', 0).astype(int).values

# Select/rename columns uniformly for concatenation
def normalize_df(df, source_type_label):
    # df must contain: source_type, cmdb_id, name, global_p95, max_value_in_window, anomaly_count, earliest_anomaly_timestamp, consecutive_run_len
    out = pd.DataFrame({
        'source_type': df.get('source_type', source_type_label),
        'cmdb_id': df['cmdb_id'],
        'name': df['name'],
        'global_p95': df.get('global_p95', np.nan),
        'max_value_in_window': df.get('max_value_in_window', np.nan),
        'anomaly_count': df.get('anomaly_count', 0).astype(int),
        'earliest_anomaly_timestamp': df.get('earliest_anomaly_timestamp', None),
        'consecutive_run_len': df.get('consecutive_run_len', 0).astype(int)
    })
    # compute percent_breach safely
    def pct(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp) or gp == 0 or pd.isna(mv):
            return np.nan
        return (mv - gp) / gp * 100.0
    out['percent_breach'] = out.apply(pct, axis=1)
    return out

trace_norm = normalize_df(trace_cands, 'trace')
metric_heap_norm = normalize_df(metric_heap, 'metric')
# For mysql_cands, ensure column mapping: mysql_cands uses 'name', 'global_p95', 'max_value_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'consecutive_run_len'
mysql_norm = normalize_df(mysql_cands, 'metric')

# 4) Concatenate all candidates and pick top 10 by percent_breach desc, tie by anomaly_count desc
all_cands = pd.concat([trace_norm, metric_heap_norm, mysql_norm], ignore_index=True, sort=False)

# Keep compact columns and sort
all_cands['percent_breach'] = all_cands['percent_breach'].astype(float)
result = all_cands.sort_values(['percent_breach','anomaly_count'], ascending=[False, False]).head(10)

# Round numeric columns for compactness
result['global_p95'] = result['global_p95'].map(lambda x: np.nan if pd.isna(x) else float(x))
result['max_value_in_window'] = result['max_value_in_window'].map(lambda x: np.nan if pd.isna(x) else float(x))
result['percent_breach'] = result['percent_breach'].map(lambda x: np.nan if pd.isna(x) else round(x, 4))

# Final compact table columns order
candidate_table = result[['source_type','cmdb_id','name','global_p95','max_value_in_window','percent_breach','anomaly_count','earliest_anomaly_timestamp','consecutive_run_len']].reset_index(drop=True)

# Return the compact ranked candidate table (top 10)
candidate_table

```
Out[6]:
```
```python
# Summarize the ranked candidate faults (top 10) in plain English using existing variable candidate_table.
if 'candidate_table' not in globals():
    raise RuntimeError("candidate_table not found in kernel. Run the candidate selection step first.")

df = candidate_table.copy()

lines = []
lines.append("Ranked candidate faults (top 10) for incident window 2021-03-24 19:00–19:30 UTC+8:")
lines.append(f"- Total candidates shown: {len(df)} (sorted by percent_breach desc).")
lines.append("- Key summary (source_type | cmdb_id | name | %breach | anomaly_count | earliest_anomaly_timestamp | longest_consecutive_run):")

for _, r in df.iterrows():
    src = r['source_type']
    cm = r['cmdb_id']
    name = r['name']
    pb = f"{r['percent_breach']:.3f}%" if pd.notna(r['percent_breach']) else "NA"
    ac = int(r['anomaly_count'])
    et = r['earliest_anomaly_timestamp'] if pd.notna(r['earliest_anomaly_timestamp']) else "None"
    run = int(r['consecutive_run_len'])
    lines.append(f"  - {src} | {cm} | {name} | {pb} | anomalies={ac} | first={et} | run_len={run}")

lines.append("")
lines.append("Observations and interpretation:")
lines.append("- The top-ranked candidates are all trace-series from IG01 and IG02 (to various Tomcat nodes), not metric HeapMemory entries.")
lines.append("- The largest percent breaches are for trace duration p95 metrics (e.g., trace.to_Tomcat01.duration_p95) with percent_breach up to ~106% (i.e., >2x the usual p95).")
lines.append("- Each candidate has anomaly_count = 2 but consecutive_run_len = 1, indicating isolated spikes (two separate minutes) rather than sustained consecutive runs.")
lines.append("- Timing: many earliest anomalies cluster around 19:03:00+0800 (one around 19:15 for to_Tomcat04), pointing to transient latency spikes originating at IG01/IG02 towards Tomcat services.")
lines.append("- Recommendation: prioritize investigating network/latency causes or request bursts at the ingress gateways (IG01/IG02) around 19:03, and check Tomcat endpoints for transient overload; HeapMemory and Mysql01 metrics appear as small isolated breaches and are lower priority compared to large trace p95 spikes.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source_type cmdb_id                             name  global_p95  max_value_in_window  percent_breach  anomaly_count earliest_anomaly_timestamp  consecutive_run_len
0       trace    IG01   trace.to_Tomcat01.duration_p95    2.942775             6.079800        106.6009              2   2021-03-24 19:03:00+0800                    1
1       trace    IG02   trace.to_Tomcat01.duration_p95    4.655150             8.974700         92.7908              2   2021-03-24 19:03:00+0800                    1
2       trace    IG02          trace.self.duration_p95    3.588875             6.895200         92.1271              2   2021-03-24 19:03:00+0800                    1
3       trace    IG01   trace.to_Tomcat04.duration_p95    2.960500             5.459000         84.3945              2   2021-03-24 19:15:00+0800                    1
4       trace    IG01          trace.self.duration_p95    3.278200             6.041000         84.2780              2   2021-03-24 19:03:00+0800                    1
5       trace    IG02  trace.to_Tomcat01.duration_mean    0.680809             1.036029         52.1762              2   2021-03-24 19:03:00+0800                    1
6       trace    IG01  trace.to_Tomcat03.duration_mean    0.751712             1.089500         44.9359              2   2021-03-24 19:03:00+0800                    1
7       trace    IG01  trace.to_Tomcat01.duration_mean    0.555889             0.765915         37.7821              2   2021-03-24 19:03:00+0800                    1
8       trace    IG02   trace.to_Tomcat02.duration_p95    3.036850             4.098300         34.9523              2   2021-03-24 19:03:00+0800                    1
9       trace    IG01   trace.to_Tomcat03.duration_p95    4.783675             6.092600         27.3623              2   2021-03-24 19:03:00+0800                    1```
```